In [ ]:
# Nayi modern google-genai library install kar rahe hain
!pip install -q google-generativeai
!pip install -q --upgrade google-genai pypdf

import os
from pypdf import PdfReader
import json

print("Nayi Google GenAI modern library successfully install ho gayi hai!")

Nayi Google GenAI modern library successfully install ho gayi hai!


In [ ]:
!pip install -U -q google-generativeai  # it is use tio test gemini

In [ ]:
# these lines are used to check the gemini model in working
import google.generativeai as genai

# Apni nayi wali API key yahan dalein
GOOGLE_API_KEY = "AQ.Ab8RN6ITdUCUh1bVCdf8MZxA2fr6c1hHP7SpN6oZK5xIaM54fw"
genai.configure(api_key=GOOGLE_API_KEY)

print("Testing connection with Gemini...")
model = genai.GenerativeModel('gemini-4-hydrogen')
response = model.generate_content("Say hello")
print("Response received successfully!")
print(response.text)

Testing connection with Gemini...
Response received successfully!
Hello! How can I help you today?


Step 2: Resume Upload & Text Extraction Function
Ab hum ek aisa code likhenge jo  Colab mein hi ek button dega jisse koi bhi sample PDF resume upload kar sakenge, aur
 code us PDF ka sara text read kar lega.

 ismeh hamneh ocr ka use kia hai yeh ocr kya hrta hai yeh image seh characters nikal k unko likhta hai pahale ham siraf pdf seh data nikal rahe thay pr usmeh problem aarahi thi ki joh pdf scan ki hui thi voh data nahi nikal parahi thi pr ab scaned vali bhi nikalegi data

In [ ]:
# Installing required libraries for PDF text extraction and OCR (Optical Character Recognition)
!pip install -q pdfplumber pdf2image pytesseract
!apt-get install -y poppler-utils tesseract-ocr > /dev/null

from google.colab import files
import pdfplumber
from pdf2image import convert_from_path
import pytesseract

def upload_and_extract_pdf():
    print("Please upload your sample resume (PDF format):")
    uploaded = files.upload()

    if not uploaded:
        print("No file uploaded.")
        return None

    pdf_filename = list(uploaded.keys())[0]
    print(f"\nSuccessfully uploaded: {pdf_filename}")

    extracted_text = ""

    # Step A: Try standard text extraction using pdfplumber
    with pdfplumber.open(pdf_filename) as pdf:
        for page in pdf.pages:
            text = page.extract_text()
            if text:
                extracted_text += text + "\n"

    # Step B: If text extraction yields almost 0 characters, use OCR (for scanned/image PDFs)
    if len(extracted_text.strip()) < 50:
        print("\n[Notice]: Standard text layer not found. Running OCR engine for scanned PDF...")
        images = convert_from_path(pdf_filename)
        for i, image in enumerate(images):
            ocr_text = pytesseract.image_to_string(image)
            extracted_text += ocr_text + "\n"

    print(f"Total characters extracted: {len(extracted_text)}")

    if len(extracted_text.strip()) < 10:
        print("\n[Error]: Unable to extract text. Please ensure the PDF contains readable text or clear scans.")

    return extracted_text

# Execute the function
resume_text = upload_and_extract_pdf()

Please upload your sample resume (PDF format):


Saving Resume_software_A.pdf to Resume_software_A (1).pdf

Successfully uploaded: Resume_software_A (1).pdf

[Notice]: Standard text layer not found. Running OCR engine for scanned PDF...
Total characters extracted: 4898


Step 3: LLM (Gemini) se Resume Parsing & Structured Data Extraction
Ab hamare paas resume ka saara raw text aa chuka hai. Lekin raw text unstructured hota hai (jise computer ya database seedhe nahi samajh sakta). Ab hum LLM ko bolenge ki is text se Name, Email, Skills, Education, aur Projects alag karke ek clean JSON format mein de.

In [ ]:
import json
import google.generativeai as genai

# Your Gemini API Key
GOOGLE_API_KEY = "AQ.Ab8RN6ITdUCUh1bVCdf8MZxA2fr6c1hHP7SpN6oZK5xIaM54fw"
genai.configure(api_key=GOOGLE_API_KEY)

def parse_resume_with_gemini(text):
    prompt = f"""
    You are an expert AI Resume Parser. Analyze the resume text provided below and extract the information strictly into a valid JSON format matching the specified structure.
    Do not include any markdown formatting wrappers like ```json or ``` in your response, just return the raw valid JSON string.

    Required JSON Structure:
    {{
      "full_name": "Candidate's full name",
      "email": "Email address",
      "phone": "Phone number",
      "skills": ["Skill 1", "Skill 2", ...],
      "education": [
        {{"degree": "Degree name", "institution": "College or University", "year": "Passing year"}}
      ],
      "projects": [
        {{"title": "Project name", "description": "Short description of the project"}}
      ]
    }}

    Resume Text:
    {text}
    """

    # Pure Gemini AI model call
    model = genai.GenerativeModel('gemini-4-hydrogen')
    response = model.generate_content(prompt)
    return response.text

# Execute parsing if resume_text exists
if 'resume_text' in locals() and resume_text:
    print("Parsing resume text using Pure Gemini AI (gemini-3.8-flash)...")
    raw_response = parse_resume_with_gemini(resume_text)

    # Clean markdown markers
    cleaned_response = raw_response.replace("```json", "").replace("```", "").strip()

    try:
        parsed_data = json.loads(cleaned_response)
        print("\n[Success]: Resume successfully parsed into 'parsed_data' dictionary!")
    except json.JSONDecodeError:
        print("\n[Error]: Could not parse response as raw JSON. Raw output:")
        print(raw_response)
else:
    print("[Error]: 'resume_text' found empty. Please run Cell 1 first.")

Parsing resume text using Pure Gemini AI (gemini-3.8-flash)...

[Success]: Resume successfully parsed into 'parsed_data' dictionary!


parsed data ko show krne k lia


In [ ]:
if 'parsed_data' in locals() and parsed_data:
    print("\n" + "="*50)
    print("--- PURE GEMINI PARSED CANDIDATE PROFILE ---")
    print("="*50)
    print(f"Name      : {parsed_data.get('full_name', 'N/A')}")
    print(f"Email     : {parsed_data.get('email', 'N/A')}")
    print(f"Phone     : {parsed_data.get('phone', 'N/A')}")
    print(f"\nSkills    : {', '.join(parsed_data.get('skills', []))}")

    print("\nEducation:")
    for edu in parsed_data.get('education', []):
        print(f" - {edu.get('degree')} at {edu.get('institution')} ({edu.get('year')})")

    print("\nProjects:")
    for proj in parsed_data.get('projects', []):
        print(f" - {proj.get('title')}: {proj.get('description')}")
    print("="*50)
else:
    print("[Error]: No parsed data found to display.")


--- PURE GEMINI PARSED CANDIDATE PROFILE ---
Name      : ANMOL SHARMA
Email     : theanmolsharmaa@gmail.com
Phone     : 9419624188

Skills    : Python, SQL, Machine Learning, Deep Learning, React.js, Next.js, Node.js, RESTful APIs, Hugging Face Transformers, n8n, Scikit-Learn, PyTorch, TensorFlow, Java, Kotlin, Librosa, NumPy, Pandas

Education:
 - Bachelor of technology (computer science engineering) at Galgotias college of engineering and technology (2024)
 - Senior Secondary Education in PCM at Sp smart higher secondary school (2020)

Projects:
 - ZeroWait Intelligent Automation Platform: Android application with an automated workflow backend built using WebView, native Java/Kotlin, Android Alarm Manager, and n8n webhooks for background task execution and data syncing, published on the Google Play Store.
 - Custom Text Generation Transformer: Built and trained a mini Encoder-Decoder Transformer model from scratch in PyTorch featuring multi-head self-attention, positional encodings,

SQLite Database Me Save Karne Ka Code

In [ ]:
import sqlite3
import json

# Step 1: Connect to a local SQLite database (yeh ek local file bana dega 'candidate_database.db')
conn = sqlite3.connect('candidate_database.db')
cursor = conn.cursor()

# Step 2: Create a table for storing candidate profiles
cursor.execute('''
    CREATE TABLE IF NOT EXISTS candidates (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        full_name TEXT,
        email TEXT,
        phone TEXT,
        skills TEXT,
        education TEXT,
        projects TEXT
    )
''')
conn.commit()

# Step 3: Insert the parsed data into the database
# Hum list aur dictionaries ko JSON string mein convert kar rahe hain taaki SQLite mein safely store ho sake
if 'parsed_data' in locals():
    cursor.execute('''
        INSERT INTO candidates (full_name, email, phone, skills, education, projects)
        VALUES (?, ?, ?, ?, ?, ?)
    ''', (
        parsed_data.get('full_name', 'N/A'),
        parsed_data.get('email', 'N/A'),
        parsed_data.get('phone', 'N/A'),
        json.dumps(parsed_data.get('skills', [])),
        json.dumps(parsed_data.get('education', [])),
        json.dumps(parsed_data.get('projects', []))
    ))
    conn.commit()
    print("Candidate profile successfully SQLite database mein save ho gayi hai!")

# Step 4: Verify by reading data back from the database
cursor.execute('SELECT * FROM candidates')
rows = cursor.fetchall()

print("\n" + "="*50)
print(f"--- DATA FROM SQLITE DATABASE (Total Records: {len(rows)}) ---")
print("="*50)
for row in rows:
    print(f"ID       : {row[0]}")
    print(f"Name     : {row[1]}")
    print(f"Email    : {row[2]}")
    print(f"Phone    : {row[3]}")
    print(f"Skills   : {json.loads(row[4])}")
print("="*50)

# Close connection
conn.close()

Candidate profile successfully SQLite database mein save ho gayi hai!

--- DATA FROM SQLITE DATABASE (Total Records: 4) ---
ID       : 1
Name     : ANMOL SHARMA
Email    : theanmolsharmaa@gmail.com
Phone    : 9419624188
Skills   : ['Python', 'Streamlit', 'SQLite', 'API Integration']
ID       : 2
Name     : SANCHIT SHARMA
Email    : sanchitsharma8721@gmail.com
Phone    : +91 6005477828
Skills   : ['Python', 'Streamlit', 'SQLite', 'API Integration']
ID       : 3
Name     : SANCHIT SHARMA
Email    : sanchitsharma8721@gmail.com
Phone    : +91 6005477828
Skills   : ['Python', 'Streamlit', 'SQLite', 'API Integration']
ID       : 4
Name     : SANCHIT SHARMA
Email    : sanchitsharma8721@gmail.com
Phone    : +91 6005477828
Skills   : ['Behavioral Analysis', 'Team Dynamics', 'Intercultural Communication', 'Conflict Resolution', 'Administrative Coordination', 'Policy Evaluation', 'Project Tracking', 'Vendor Management', 'Content Strategy', 'Qualitative Research', 'Data Synthesis', 'Technical Comm

Making app for better interface means front end


In [ ]:
# Step 1: Streamlit aur localtunnel install karein
!pip install -q streamlit pyngrok pypdf

# Step 2: Colab ke andar hi app.py file create karein
app_code = """
import streamlit as st
import json
import google.generativeai as genai
from pypdf import PdfReader
import sqlite3

st.set_page_config(page_title="AI Internship Assistant", layout="centered")

st.title("🚀 AI-Powered Internship & Career Assistant")
st.subheader("Milestone 1: Candidate Profile & Resume Parser")

api_key = st.text_input("Enter your Gemini API Key", type="password")
uploaded_file = st.file_uploader("Upload your Resume (PDF)", type=["pdf"])

if uploaded_file and api_key:
    genai.configure(api_key=api_key)

    with st.spinner("Extracting text and parsing with AI..."):
        with open("temp_resume.pdf", "wb") as f:
            f.write(uploaded_file.getbuffer())

        reader = PdfReader("temp_resume.pdf")
        resume_text = ""
        for page in reader.pages:
            resume_text += page.extract_text() or ""

        model = genai.GenerativeModel('gemini-4-hydrogen')
        prompt = f\"\"\"
        You are an expert AI Resume Parser. Analyze the resume text below and extract into valid JSON:
        {{
          "full_name": "Name",
          "email": "Email",
          "phone": "Phone",
          "skills": ["Skill 1", "Skill 2"],
          "education": [{{"degree": "Degree", "institution": "College", "year": "Year"}}],
          "projects": [{{"title": "Title", "description": "Desc"}}]
        }}
        Resume Text: {resume_text}
        \"\"\"

        response = model.generate_content(prompt)
        cleaned = response.text.replace("```json", "").replace("```", "").strip()

        try:
            data = json.loads(cleaned)
            st.success("Resume successfully parsed!")
            st.json(data)

            conn = sqlite3.connect('candidate_database.db')
            cursor = conn.cursor()
            cursor.execute('''
                CREATE TABLE IF NOT EXISTS candidates (
                    id INTEGER PRIMARY KEY AUTOINCREMENT,
                    full_name TEXT, email TEXT, phone TEXT, skills TEXT, education TEXT, projects TEXT
                )
            ''')
            cursor.execute('INSERT INTO candidates (full_name, email, phone, skills, education, projects) VALUES (?, ?, ?, ?, ?, ?)',
                           (data.get('full_name'), data.get('email'), data.get('phone'),
                            json.dumps(data.get('skills')), json.dumps(data.get('education')), json.dumps(data.get('projects'))))
            conn.commit()
            conn.close()
            st.info("Profile successfully saved to SQLite database!")

        except Exception as e:
            st.error(f"Error parsing JSON: {e}")
            st.text(response.text)
"""

with open("app.py", "w") as f:
    f.write(app_code)

print("app.py successfully created inside Colab!")

app.py successfully created inside Colab!


for running at port


In [ ]:
# Cloudflare tunnel download aur run karne ke liye
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!streamlit run app.py & ./cloudflared-linux-amd64 tunnel --url http://localhost:8501

2026-11-14T16:31:30Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-11-14T16:31:30Z INF Requesting new quick Tunnel on trycloudflare.com...


2026-11-14 16:31:33.055 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://136.90.30.114:8501

2026-11-14T16:31:34Z INF +-------------------------------------------------------------